# LJ Dev Commerce - Phase 4

# III. Category ETL

## 1. Extract

Load the raw Category dataset from the Product Catalog source system.

The raw dataset uses source-system field names and will be profiled before
any cleaning or transformation is applied.

In [ ]:
import pandas as pd

In [2]:
# Load raw Category data
from pathlib import Path
import sys

# Read the raw Category CSV
product_catalog_categories_raw = pd.read_csv(
    "../data/03_Product_Catalog/product_catalog_categories.csv"
)

product_catalog_categories_raw.head()

,CategoryCode,CategoryName,Description,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,CAT001,Laptops,Portable computers,True,2026-07-01,admin,2026-07-15,admin
1,CAT002,Smartphones,Mobile phones,True,2026-07-01,admin,2026-07-15,admin
2,CAT003,Tablets,Portable tablets,True,2026-07-02,admin,2026-07-15,admin
3,CAT004,Monitors,Computer displays,True,2026-07-02,admin,2026-07-14,admin
4,CAT005,accessories,Computer and mobile accessories,True,2026-07-03,admin,2026-07-15,admin


### 1.2 Initial Data Inspection

Review the raw Category dataset to understand its structure, row count,
column names, data types, and initial sample records before applying any
cleaning or transformation.

In [3]:
# Initial Category data inspection

print("Rows:", len(product_catalog_categories_raw))
print("Columns:", len(product_catalog_categories_raw.columns))

print("\nColumn names:")
print(product_catalog_categories_raw.columns.tolist())

print("\nData types:")
print(product_catalog_categories_raw.dtypes)

print("\nMissing values:")
print(product_catalog_categories_raw.isna().sum())

display(product_catalog_categories_raw.head())

Rows: 6
Columns: 8

Column names:
['CategoryCode', 'CategoryName', 'Description', 'ActiveFlag', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Data types:
CategoryCode      object
CategoryName      object
Description       object
ActiveFlag          bool
CreatedOn         object
CreatedByUser     object
ModifiedOn        object
ModifiedByUser    object
dtype: object

Missing values:
CategoryCode      0
CategoryName      0
Description       0
ActiveFlag        0
CreatedOn         0
CreatedByUser     0
ModifiedOn        0
ModifiedByUser    0
dtype: int64


,CategoryCode,CategoryName,Description,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,CAT001,Laptops,Portable computers,True,2026-07-01,admin,2026-07-15,admin
1,CAT002,Smartphones,Mobile phones,True,2026-07-01,admin,2026-07-15,admin
2,CAT003,Tablets,Portable tablets,True,2026-07-02,admin,2026-07-15,admin
3,CAT004,Monitors,Computer displays,True,2026-07-02,admin,2026-07-14,admin
4,CAT005,accessories,Computer and mobile accessories,True,2026-07-03,admin,2026-07-15,admin


### 1.3 Import and Reload Reusable Profiler

Import the reusable `data_profiler_v1` module from the project profiler package.

The profiler is reloaded to ensure that the notebook uses the current version
of the reusable profiling logic and configuration.


In [4]:
# Import and reload the current reusable profiler
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
from profiler import data_profiler_v1 as profiler

importlib.reload(profiler)

print("Profiler:", profiler.__file__)
print("Configuration:", profiler.DEFAULT_CONFIG)

Profiler: c:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\profiler\data_profiler_v1.py
Configuration: {'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


### 1.4 Profile Raw Category Dataset

Run the reusable `data_profiler_v1` against the raw Category dataset to identify
potential data-quality issues before applying any cleaning or transformation.

The profiling results will be reviewed to determine which issues can be safely
and deterministically automated and which require business-rule review.


In [5]:
# Run the complete profiler against the untouched raw Category dataset
profile_results = profiler.profile_dataset(product_catalog_categories_raw)

print("Profiler sections:")
print(list(profile_results.keys()))
print("Category profiling completed.")

Profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']
Category profiling completed.


### 1.5 Review Profiler Results

Review the profiling results generated from the untouched raw Category dataset.

The `issues` output will be examined first to identify data-quality problems.
Each identified issue will then be assessed to determine whether it is safe
and deterministic to automate or requires a business-rule decision.


In [6]:
# Review detected Category data-quality issues

category_issues = profile_results["issues"]

display(category_issues)

,Column,IssueType,Severity,Count,Description
0,CategoryName,Whitespace,Low,1,Leading or trailing whitespace detected.


### 1.5.1 Independent Analyst Review

The reusable profiler is the first automated quality check, but it is not the sole data-readiness check.

Perform an independent review of the raw Category dataset for issues that may not be reported as profiler "issues", including semantic data types, date validity and consistency, key format and uniqueness, boolean integrity, blank text, and other structural/business-rule concerns.

The review is performed before transformation so that findings can be separated from profiler findings and documented explicitly.

In [7]:
# Independent analyst review — Category data readiness

date_columns = ["CreatedOn", "ModifiedOn"]
text_columns = [
    "CategoryCode",
    "CategoryName",
    "Description",
    "CreatedByUser",
    "ModifiedByUser",
]

print("Profiler semantic field types:")
print(profile_results["field_types"])

print("\nCurrent pandas dtypes:")
print(product_catalog_categories_raw.dtypes)

# Date parsing, format, and future-date checks
for col in date_columns:
    parsed = pd.to_datetime(product_catalog_categories_raw[col], errors="coerce")

    invalid_count = parsed.isna().sum()
    iso_format_count = (
        product_catalog_categories_raw[col]
        .astype(str)
        .str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    ).sum()

    future_count = (parsed > pd.Timestamp.today().normalize()).sum()

    print(f"\n{col}:")
    print("  Invalid dates:", invalid_count)
    print("  ISO-format dates:", iso_format_count, "of", len(product_catalog_categories_raw))
    print("  Unexpected future dates:", future_count)

# Created/Modified logical consistency
created = pd.to_datetime(product_catalog_categories_raw["CreatedOn"], errors="coerce")
modified = pd.to_datetime(product_catalog_categories_raw["ModifiedOn"], errors="coerce")

print(
    "\nModifiedOn earlier than CreatedOn:",
    (modified < created).sum()
)

# Key integrity
print("\nCategoryCode unique:", product_catalog_categories_raw["CategoryCode"].is_unique)
print(
    "Invalid CategoryCode format:",
    (~product_catalog_categories_raw["CategoryCode"].astype(str).str.match(
        r"^CAT\d{3}$", na=False
    )).sum()
)

# Boolean integrity
print("\nActiveFlag dtype:", product_catalog_categories_raw["ActiveFlag"].dtype)
print(
    "ActiveFlag invalid values:",
    (~product_catalog_categories_raw["ActiveFlag"].isin([True, False])).sum()
)

# Blank/whitespace-only text values
print("\nBlank/whitespace-only text values:")
for col in text_columns:
    count = product_catalog_categories_raw[col].astype("string").str.strip().eq("").sum()
    print(f"  {col}: {count}")
        

Profiler semantic field types:
CategoryCode       identifier
CategoryName             text
Description              text
ActiveFlag            boolean
CreatedOn                date
CreatedByUser     categorical
ModifiedOn               date
ModifiedByUser    categorical
Name: DetectedFieldType, dtype: object

Current pandas dtypes:
CategoryCode      object
CategoryName      object
Description       object
ActiveFlag          bool
CreatedOn         object
CreatedByUser     object
ModifiedOn        object
ModifiedByUser    object
dtype: object

CreatedOn:
  Invalid dates: 0
  ISO-format dates: 6 of 6
  Unexpected future dates: 0

ModifiedOn:
  Invalid dates: 0
  ISO-format dates: 6 of 6
  Unexpected future dates: 0

ModifiedOn earlier than CreatedOn: 0

CategoryCode unique: True
Invalid CategoryCode format: 0

ActiveFlag dtype: bool
ActiveFlag invalid values: 0

Blank/whitespace-only text values:
  CategoryCode: 0
  CategoryName: 0
  Description: 0
  CreatedByUser: 0
  ModifiedByUser: 0


### 1.6 Profiling Findings Summary

The raw Category dataset contains 6 records and 8 columns. The profiler identified one data-quality issue:

| Column         | Issue                       | Severity | Count | Decision |
| -------------- | --------------------------- | -------- | ----: | -------- |
| `CategoryName` | Leading/trailing whitespace | Low      |     1 | Fix      |

The value `accessories` was also reviewed for capitalization consistency. Cross-dataset validation confirmed that related Product records reference the category through `CategoryCode = CAT005`, and no authoritative conflicting `CategoryName` value or capitalization rule was identified.

Therefore, the lowercase value will be preserved. Only the confirmed leading/trailing whitespace issue will be corrected.


### 1.7 Profiling & Independent Review Decision

The profiler identified one data-quality issue in `CategoryName`:

- Leading/trailing whitespace — safe and deterministic to fix.

An independent analyst review also identified that `CreatedOn` and `ModifiedOn` are currently stored by pandas as `object` values even though they contain date fields. The values must therefore be validated and converted to proper datetime types before database loading.

The following decisions apply:

- `CategoryName` whitespace → **Fix**
- `CreatedOn` / `ModifiedOn` datatype → **Convert to datetime after validation**
- `accessories` capitalization → **Preserve**
- `ActiveFlag` boolean representation → **Preserve**
- `CategoryCode` values → **Preserve**

The raw Category dataset will remain unchanged. All approved transformations will be applied only to `product_catalog_categories_clean`.


### 1.8 Data Transformation & Cleaning

Create a separate transformation dataframe from the raw Category dataset.

Only the approved transformation identified during profiling will be applied:
leading and trailing whitespace will be removed from `CategoryName`.

The raw Category dataset will remain unchanged to preserve source-data auditability and enable before-and-after validation.


In [8]:
# Create transformation dataset

product_catalog_categories_clean = product_catalog_categories_raw.copy()

print("Raw rows:", len(product_catalog_categories_raw))
print("Transformation rows:", len(product_catalog_categories_clean))

Raw rows: 6
Transformation rows: 6


In [9]:
# Capture the raw Category dataset before any transformation

product_catalog_categories_raw_snapshot = product_catalog_categories_raw.copy()

print(
    "Raw snapshot captured:",
    product_catalog_categories_raw.equals(product_catalog_categories_raw_snapshot)
)

Raw snapshot captured: True


In [10]:
# Inspect CategoryName values with leading/trailing whitespace

category_whitespace = product_catalog_categories_clean[
    product_catalog_categories_clean["CategoryName"]
    != product_catalog_categories_clean["CategoryName"].str.strip()
][["CategoryCode", "CategoryName"]]

display(category_whitespace)

,CategoryCode,CategoryName
4,CAT005,accessories


### 1.8.1 Remove Leading and Trailing Whitespace

Remove leading and trailing whitespace from the `CategoryName` field.

This is a safe and deterministic transformation identified during profiling. No other CategoryName values will be modified.


In [11]:
# Remove leading and trailing whitespace from CategoryName

product_catalog_categories_clean["CategoryName"] = product_catalog_categories_clean["CategoryName"].str.strip()

print("CategoryName whitespace cleanup completed.")

CategoryName whitespace cleanup completed.


### 1.8.2 Standardize Date Fields

Convert `CreatedOn` and `ModifiedOn` from source text values to pandas datetime values.

This is a safe deterministic transformation because the independent review confirmed that the values are valid ISO-formatted dates with no invalid or logically inconsistent dates.

No date values will be changed; only their dataframe datatype will be standardized.

In [12]:
# Convert Category date fields to proper datetime types

product_catalog_categories_clean["CreatedOn"] = pd.to_datetime(
    product_catalog_categories_clean["CreatedOn"],
    errors="raise"
)

product_catalog_categories_clean["ModifiedOn"] = pd.to_datetime(
    product_catalog_categories_clean["ModifiedOn"],
    errors="raise"
)

print("Category date fields converted to datetime.")
print(product_catalog_categories_clean[["CreatedOn", "ModifiedOn"]].dtypes)

Category date fields converted to datetime.
CreatedOn     datetime64[ns]
ModifiedOn    datetime64[ns]
dtype: object


### 1.9 Transformation Validation

Validate the approved Category transformations before final database-readiness validation.

The checks confirm:

- `CategoryName` whitespace was removed.
- `CreatedOn` and `ModifiedOn` are now proper datetime values.
- Row count remains unchanged.
- The raw dataset remains unchanged.
- Only approved transformations were applied.

In [13]:
# Confirm no leading/trailing whitespace remains in CategoryName

remaining_whitespace = product_catalog_categories_clean[
    product_catalog_categories_clean["CategoryName"]
    != product_catalog_categories_clean["CategoryName"].str.strip()
]

print("Remaining CategoryName whitespace issues:", len(remaining_whitespace))
display(remaining_whitespace)

Remaining CategoryName whitespace issues: 0


,CategoryCode,CategoryName,Description,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser


In [14]:
# Validate row count was preserved

print("Raw rows:", len(product_catalog_categories_raw))
print("Clean rows:", len(product_catalog_categories_clean))
print("Row count validation:", len(product_catalog_categories_raw) == len(product_catalog_categories_clean))

Raw rows: 6
Clean rows: 6
Row count validation: True


In [15]:
# Validate standardized date fields

print("CreatedOn dtype:", product_catalog_categories_clean["CreatedOn"].dtype)
print("ModifiedOn dtype:", product_catalog_categories_clean["ModifiedOn"].dtype)

print(
    "Invalid CreatedOn dates:",
    product_catalog_categories_clean["CreatedOn"].isna().sum()
)

print(
    "Invalid ModifiedOn dates:",
    product_catalog_categories_clean["ModifiedOn"].isna().sum()
)

print(
    "ModifiedOn earlier than CreatedOn:",
    (
        product_catalog_categories_clean["ModifiedOn"]
        < product_catalog_categories_clean["CreatedOn"]
    ).sum()
)

print(
    "Unexpected future CreatedOn dates:",
    (
        product_catalog_categories_clean["CreatedOn"]
        > pd.Timestamp.today().normalize()
    ).sum()
)

print(
    "Unexpected future ModifiedOn dates:",
    (
        product_catalog_categories_clean["ModifiedOn"]
        > pd.Timestamp.today().normalize()
    ).sum()
)

CreatedOn dtype: datetime64[ns]
ModifiedOn dtype: datetime64[ns]
Invalid CreatedOn dates: 0
Invalid ModifiedOn dates: 0
ModifiedOn earlier than CreatedOn: 0
Unexpected future CreatedOn dates: 0
Unexpected future ModifiedOn dates: 0


In [16]:
# Confirm the raw Category dataset remained unchanged

print(
    "Raw dataset preserved:",
    product_catalog_categories_raw.equals(product_catalog_categories_raw_snapshot)
)

Raw dataset preserved: True


### 1.10 Final Category Validation

Perform final validation of the transformed Category dataset before database preparation.

The final validation confirms structural integrity, missing-value status, key integrity, text-cleaning results, semantic datatypes, boolean integrity, date integrity, and preservation of the raw source dataset.

The Category dataset will proceed to PostgreSQL only after all required validation checks pass.

In [17]:
# Final Category dataset validation

print("Rows:", len(product_catalog_categories_clean))
print("Columns:", len(product_catalog_categories_clean.columns))

print("\nMissing values:")
print(product_catalog_categories_clean.isna().sum())

print(
    "\nDuplicate CategoryCode values:",
    product_catalog_categories_clean["CategoryCode"].duplicated().sum()
)

print(
    "CategoryCode format violations:",
    (~product_catalog_categories_clean["CategoryCode"].astype(str).str.match(
        r"^CAT\d{3}$", na=False
    )).sum()
)

print(
    "\nRemaining CategoryName whitespace issues:",
    (
        product_catalog_categories_clean["CategoryName"]
        != product_catalog_categories_clean["CategoryName"].str.strip()
    ).sum()
)

print("\nData types:")
print(product_catalog_categories_clean.dtypes)

print("\nActiveFlag dtype:", product_catalog_categories_clean["ActiveFlag"].dtype)
print(
    "ActiveFlag invalid values:",
    (~product_catalog_categories_clean["ActiveFlag"].isin([True, False])).sum()
)

print("\nColumn names:")
print(product_catalog_categories_clean.columns.tolist())

print(
    "\nRaw dataset preserved:",
    product_catalog_categories_raw.equals(product_catalog_categories_raw_snapshot)
)

Rows: 6
Columns: 8

Missing values:
CategoryCode      0
CategoryName      0
Description       0
ActiveFlag        0
CreatedOn         0
CreatedByUser     0
ModifiedOn        0
ModifiedByUser    0
dtype: int64

Duplicate CategoryCode values: 0
CategoryCode format violations: 0

Remaining CategoryName whitespace issues: 0

Data types:
CategoryCode              object
CategoryName              object
Description               object
ActiveFlag                  bool
CreatedOn         datetime64[ns]
CreatedByUser             object
ModifiedOn        datetime64[ns]
ModifiedByUser            object
dtype: object

ActiveFlag dtype: bool
ActiveFlag invalid values: 0

Column names:
['CategoryCode', 'CategoryName', 'Description', 'ActiveFlag', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Raw dataset preserved: True


### 1.10.1 Approved Transformation Comparison

Compare the raw and transformed Category datasets to confirm that only the approved changes were introduced.

Expected changes:

- `CategoryName`: leading/trailing whitespace removed for the affected record.
- `CreatedOn`: datatype changed from source text/object to datetime.
- `ModifiedOn`: datatype changed from source text/object to datetime.

No business values should be changed.

In [18]:
# Compare raw and transformed Category values
# Normalize date columns to the same string representation for value comparison.
# This separates value changes from intentional datatype changes.

raw_value_check = product_catalog_categories_raw.copy()
clean_value_check = product_catalog_categories_clean.copy()

for col in ["CreatedOn", "ModifiedOn"]:
    raw_value_check[col] = pd.to_datetime(
        raw_value_check[col],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    clean_value_check[col] = clean_value_check[col].dt.strftime("%Y-%m-%d")

category_value_comparison = raw_value_check.compare(
    clean_value_check,
    keep_shape=False,
    keep_equal=False
)

display(category_value_comparison)

print(
    "\nBusiness-value differences detected:",
    len(category_value_comparison)
)

print("\nDatatype changes:")
print(
    pd.DataFrame({
        "Raw": product_catalog_categories_raw.dtypes,
        "Clean": product_catalog_categories_clean.dtypes
    })
)
        

CategoryName             
            self        other
4   accessories   accessories


Business-value differences detected: 1

Datatype changes:
                   Raw           Clean
CategoryCode    object          object
CategoryName    object          object
Description     object          object
ActiveFlag        bool            bool
CreatedOn       object  datetime64[ns]
CreatedByUser   object          object
ModifiedOn      object  datetime64[ns]
ModifiedByUser  object          object


In [19]:
# Final Category readiness summary

print("CreatedOn invalid:", product_catalog_categories_clean["CreatedOn"].isna().sum())
print("ModifiedOn invalid:", product_catalog_categories_clean["ModifiedOn"].isna().sum())

print(
    "ModifiedOn earlier than CreatedOn:",
    (
        product_catalog_categories_clean["ModifiedOn"]
        < product_catalog_categories_clean["CreatedOn"]
    ).sum()
)

print(
    "Future CreatedOn:",
    (
        product_catalog_categories_clean["CreatedOn"]
        > pd.Timestamp.today().normalize()
    ).sum()
)

print(
    "Future ModifiedOn:",
    (
        product_catalog_categories_clean["ModifiedOn"]
        > pd.Timestamp.today().normalize()
    ).sum()
)

print(
    "ActiveFlag invalid:",
    (~product_catalog_categories_clean["ActiveFlag"].isin([True, False])).sum()
)

print("\nExpected business-value difference: 1")
print("Actual business-value differences:", len(category_value_comparison))

CreatedOn invalid: 0
ModifiedOn invalid: 0
ModifiedOn earlier than CreatedOn: 0
Future CreatedOn: 0
Future ModifiedOn: 0
ActiveFlag invalid: 0

Expected business-value difference: 1
Actual business-value differences: 1


### 1.11 Export Clean Category Dataset

Create the clean Category CSV from the fully validated `product_catalog_categories_clean` dataframe.

The raw source file remains unchanged. This exported CSV is the approved cleaned dataset that will be carried forward to the PostgreSQL loading stage.

In [20]:
# Export the validated Category dataset as a clean CSV

clean_category_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\03_Product_Catalog\clean\product_catalog_categories_clean.csv"
)

clean_category_path.parent.mkdir(parents=True, exist_ok=True)

product_catalog_categories_clean.to_csv(
    clean_category_path,
    index=False
)

print("Clean Category CSV created:")
print(clean_category_path.resolve())

print("\nRows exported:", len(product_catalog_categories_clean))
print("Columns exported:", len(product_catalog_categories_clean.columns))

Clean Category CSV created:
C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\03_Product_Catalog\clean\product_catalog_categories_clean.csv

Rows exported: 6
Columns exported: 8


### 1.12 Verify Clean Category CSV

Reload the newly created clean CSV and verify that the exported file contains the same approved data as `product_catalog_categories_clean`.

This confirms that the file we will use for PostgreSQL loading was actually produced from the validated dataframe and was not accidentally altered during export.

In [21]:
# Reload the clean CSV and verify the exported dataset

product_catalog_categories_clean_csv = pd.read_csv(
    clean_category_path,
    parse_dates=["CreatedOn", "ModifiedOn"]
)

print("Clean CSV rows:", len(product_catalog_categories_clean_csv))
print("Clean dataframe rows:", len(product_catalog_categories_clean))

print(
    "Row count matches:",
    len(product_catalog_categories_clean_csv) == len(product_catalog_categories_clean)
)

print(
    "Column structure matches:",
    list(product_catalog_categories_clean_csv.columns) == list(product_catalog_categories_clean.columns)
)

print(
    "Values match validated product_catalog_categories_clean:",
    product_catalog_categories_clean_csv.equals(product_catalog_categories_clean)
)

print("\nClean CSV data types:")
print(product_catalog_categories_clean_csv.dtypes)

Clean CSV rows: 6
Clean dataframe rows: 6
Row count matches: True
Column structure matches: True
Values match validated product_catalog_categories_clean: True

Clean CSV data types:
CategoryCode              object
CategoryName              object
Description               object
ActiveFlag                  bool
CreatedOn         datetime64[ns]
CreatedByUser             object
ModifiedOn        datetime64[ns]
ModifiedByUser            object
dtype: object


### 1.12 Database-Ready Category Dataset

Prepare the validated `product_catalog_categories_clean` dataset for PostgreSQL loading.

The source Category fields are mapped to the documented PostgreSQL target columns. No business values are changed during this step; only the target database column names are applied.

The database-ready dataset will be validated before PostgreSQL loading.


In [60]:
# Create database-ready Category dataset

product_catalog_categories_db_ready = product_catalog_categories_clean.rename(columns={
    "CategoryCode": "category_id",
    "CategoryName": "category_name",
    "Description": "category_description",
    "ActiveFlag": "is_active",
    "CreatedOn": "created_date",
    "CreatedByUser": "created_by",
    "ModifiedOn": "updated_date",
    "ModifiedByUser": "updated_by"
}).copy()

print("Database-ready rows:", len(product_catalog_categories_db_ready))
print("Database-ready columns:", len(product_catalog_categories_db_ready.columns))

print("\nDatabase-ready columns:")
print(product_catalog_categories_db_ready.columns.tolist())

Database-ready rows: 6
Database-ready columns: 8

Database-ready columns:
['category_id', 'category_name', 'category_description', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']


### 1.12.1 Database-Ready Validation

Validate the database-ready Category dataset before PostgreSQL loading.

The validation confirms the expected target column structure, row count, missing values, key uniqueness, data types, and date integrity.


In [61]:
# Validate database-ready Category dataset

expected_category_columns = [
    "category_id",
    "category_name",
    "category_description",
    "is_active",
    "created_date",
    "created_by",
    "updated_date",
    "updated_by"
]

print("Column structure matches:",
      list(product_catalog_categories_db_ready.columns) == expected_category_columns)

print("\nRows:", len(product_catalog_categories_db_ready))
print("Expected rows:", len(product_catalog_categories_clean))
print("Row count matches:",
      len(product_catalog_categories_db_ready) == len(product_catalog_categories_clean))

print("\nMissing values:")
print(product_catalog_categories_db_ready.isna().sum())

print("\nDuplicate category_id values:",
      product_catalog_categories_db_ready["category_id"].duplicated().sum())

print("\nData types:")
print(product_catalog_categories_db_ready.dtypes)

Column structure matches: True

Rows: 6
Expected rows: 6
Row count matches: True

Missing values:
category_id             0
category_name           0
category_description    0
is_active               0
created_date            0
created_by              0
updated_date            0
updated_by              0
dtype: int64

Duplicate category_id values: 0

Data types:
category_id                     object
category_name                   object
category_description            object
is_active                         bool
created_date            datetime64[ns]
created_by                      object
updated_date            datetime64[ns]
updated_by                      object
dtype: object


### 1.13 PostgreSQL Connection

Establish a connection from the Python notebook to the PostgreSQL database.

The connection will be used to load the validated `product_catalog_categories_db_ready` dataset into the existing `commerce.category` target table.

No Category records are inserted at this stage. This step only establishes and verifies the database connection.


In [63]:
# Establish PostgreSQL connection

import psycopg2
from getpass import getpass

password = getpass("PostgreSQL password: ")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="lj_dev_commerce",
    user="postgres",
    password=password
)

cursor = conn.cursor()

print("PostgreSQL connection established successfully.")

PostgreSQL connection established successfully.


### 1.14 Verify Database-Ready Dataset

Before inserting records, verify that the validated `product_catalog_categories_db_ready` dataset contains the expected records and target PostgreSQL columns.

No database records are inserted during this step.


In [64]:
# Verify database-ready Category dataset before loading

print("Rows:", len(product_catalog_categories_db_ready))
print("Columns:", len(product_catalog_categories_db_ready.columns))

print("\nColumns:")
print(product_catalog_categories_db_ready.columns.tolist())

print("\nDatabase-ready Category data:")
display(product_catalog_categories_db_ready)

Rows: 6
Columns: 8

Columns:
['category_id', 'category_name', 'category_description', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']

Database-ready Category data:


,category_id,category_name,category_description,is_active,created_date,created_by,updated_date,updated_by
0,CAT001,Laptops,Portable computers,True,2026-07-01,admin,2026-07-15,admin
1,CAT002,Smartphones,Mobile phones,True,2026-07-01,admin,2026-07-15,admin
2,CAT003,Tablets,Portable tablets,True,2026-07-02,admin,2026-07-15,admin
3,CAT004,Monitors,Computer displays,True,2026-07-02,admin,2026-07-14,admin
4,CAT005,accessories,Computer and mobile accessories,True,2026-07-03,admin,2026-07-15,admin
5,CAT006,Networking,Networking equipment,True,2026-07-03,admin,2026-07-15,admin


### 1.15 Parameterized Category Insert

Insert the validated `product_catalog_categories_db_ready` records into the existing `commerce.category` PostgreSQL table using a parameterized SQL statement.

The database-ready dataset is used as the sole input. Parameterized insertion is used to keep the load controlled and safe.


In [65]:
# Insert Category records into PostgreSQL

insert_sql = """
INSERT INTO commerce.category (
    category_id,
    category_name,
    category_description,
    is_active,
    created_date,
    created_by,
    updated_date,
    updated_by
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
"""

category_rows = list(
    product_catalog_categories_db_ready.itertuples(index=False, name=None)
)

cursor.executemany(
    insert_sql,
    category_rows
)

conn.commit()

print("Category records inserted successfully.")
print("Records inserted:", cursor.rowcount)

Category records inserted successfully.
Records inserted: 6


### 1.16 Row-Count Validation

Verify that the number of Category records stored in PostgreSQL matches the validated database-ready dataset.

This confirms that all expected Category records were inserted without loss or unexpected additional records.


In [66]:
# Validate Category row count in PostgreSQL

cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.category;
""")

db_category_count = cursor.fetchone()[0]
expected_category_count = len(product_catalog_categories_db_ready)

print("Database rows:", db_category_count)
print("Expected rows:", expected_category_count)
print("Row count matches:", db_category_count == expected_category_count)

Database rows: 6
Expected rows: 6
Row count matches: True


### 1.17 Retrieve Category Records from PostgreSQL

Retrieve the loaded Category records from PostgreSQL for direct comparison against the validated `product_catalog_categories_db_ready` dataset.

This verifies that the records stored in the database match the data that was intended for loading.


In [67]:
# Retrieve loaded Category records from PostgreSQL

cursor.execute("""
    SELECT
        category_id,
        category_name,
        category_description,
        is_active,
        created_date,
        created_by,
        updated_date,
        updated_by
    FROM commerce.category
    ORDER BY category_id;
""")

category_db = cursor.fetchall()

print("Retrieved Category records:", len(category_db))

for row in category_db:
    print(row)

Retrieved Category records: 6
('CAT001', 'Laptops', 'Portable computers', True, datetime.datetime(2026, 7, 1, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('CAT002', 'Smartphones', 'Mobile phones', True, datetime.datetime(2026, 7, 1, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('CAT003', 'Tablets', 'Portable tablets', True, datetime.datetime(2026, 7, 2, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('CAT004', 'Monitors', 'Computer displays', True, datetime.datetime(2026, 7, 2, 0, 0), 'admin', datetime.datetime(2026, 7, 14, 0, 0), 'admin')
('CAT005', 'accessories', 'Computer and mobile accessories', True, datetime.datetime(2026, 7, 3, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('CAT006', 'Networking', 'Networking equipment', True, datetime.datetime(2026, 7, 3, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')


In [68]:
# Compare database records with database-ready Category dataset

category_db_columns = [
    "category_id",
    "category_name",
    "category_description",
    "is_active",
    "created_date",
    "created_by",
    "updated_date",
    "updated_by"
]

category_db_retrieved = pd.DataFrame(
    category_db,
    columns=category_db_columns
)

# Normalize datetime columns for comparison
for col in ["created_date", "updated_date"]:
    category_db_retrieved[col] = pd.to_datetime(
        category_db_retrieved[col]
    )

# Sort both datasets by category_id
expected = (
    product_catalog_categories_db_ready
    .sort_values("category_id")
    .reset_index(drop=True)
)

actual = (
    category_db_retrieved
    .sort_values("category_id")
    .reset_index(drop=True)
)

# Compare values
values_match = expected.equals(actual)

print("Expected rows:", len(expected))
print("Database rows:", len(actual))
print("Row count matches:", len(expected) == len(actual))
print("Field-level values match:", values_match)

Expected rows: 6
Database rows: 6
Row count matches: True
Field-level values match: True


### 1.19 Category Database Integrity Validation

Perform final integrity checks on the loaded Category table.

The checks confirm primary-key uniqueness, NULL integrity, date integrity, active-flag validity, and the expected record count.

These checks provide a final confirmation that the Category table is structurally and business-rule compliant after loading.


In [69]:
# Final Category database integrity validation

cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT category_id) AS distinct_category_ids,
        COUNT(*) FILTER (WHERE category_id IS NULL) AS null_category_ids,
        COUNT(*) FILTER (WHERE category_name IS NULL) AS null_category_names,
        COUNT(*) FILTER (WHERE category_description IS NULL) AS null_descriptions,
        COUNT(*) FILTER (WHERE is_active IS NULL) AS null_active_flags,
        COUNT(*) FILTER (WHERE created_date IS NULL) AS null_created_dates,
        COUNT(*) FILTER (WHERE updated_date IS NULL) AS null_updated_dates,
        COUNT(*) FILTER (WHERE is_active NOT IN (TRUE, FALSE)) AS invalid_active_flags,
        COUNT(*) FILTER (WHERE updated_date < created_date) AS invalid_date_order
    FROM commerce.category;
""")

integrity_result = cursor.fetchone()

print("Total rows:", integrity_result[0])
print("Distinct category IDs:", integrity_result[1])
print("NULL category IDs:", integrity_result[2])
print("NULL category names:", integrity_result[3])
print("NULL descriptions:", integrity_result[4])
print("NULL active flags:", integrity_result[5])
print("NULL created dates:", integrity_result[6])
print("NULL updated dates:", integrity_result[7])
print("Invalid active flags:", integrity_result[8])
print("Updated date earlier than created date:", integrity_result[9])

Total rows: 6
Distinct category IDs: 6
NULL category IDs: 0
NULL category names: 0
NULL descriptions: 0
NULL active flags: 0
NULL created dates: 0
NULL updated dates: 0
Invalid active flags: 0
Updated date earlier than created date: 0


### 1.20 Close PostgreSQL Connection

Close the PostgreSQL cursor and database connection after completing the Category load and validation steps.

This releases the database resources used by the ETL process.


In [70]:
# Close PostgreSQL cursor and connection

cursor.close()
conn.close()

print("PostgreSQL connection closed successfully.")

PostgreSQL connection closed successfully.
